In [35]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [36]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 32           # EMNIST is 28x28; pad/resize to 32x32 (divisible by 8)
BATCH_SIZE = 128
EPOCHS = 10              # bump up for better sample quality
LR = 2e-4
T = 200                  # total diffusion timesteps (per the assignment)
BASE_CH = 32
DATA_ROOT = "./data"
OUT_DIR = "./outputs_ddpm"
os.makedirs(OUT_DIR, exist_ok=True)
SNAPSHOT_STEPS = [200, 150, 100, 50, 0]

torch.manual_seed(0)

In [37]:
def get_dataloader(batch_size=BATCH_SIZE, img_size=IMG_SIZE):
    transform = transforms.Compose([
        transforms.Lambda(lambda img: img.rotate(-90).transpose(0)),  # fix EMNIST orientation
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),                        # -> [0, 1]
        transforms.Lambda(lambda x: x * 2.0 - 1.0),    # -> [-1, 1]
    ])

    dataset = datasets.EMNIST(
        root=DATA_ROOT,
        split="letters",
        train=True,
        download=True,
        transform=transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        drop_last=True,
    )
    return loader, dataset

In [38]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        freq = math.log(10000) / (half_dim - 1)
        freq = torch.exp(torch.arange(half_dim, device=device) * -freq)
        args = t[:, None].float() * freq[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb

In [39]:
class TimeMLP(nn.Module):
    def __init__(self, dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            SinusoidalTimeEmbedding(dim),
            nn.Linear(dim, out_dim),
            nn.SiLU(),
            nn.Linear(out_dim, out_dim),
        )

    def forward(self, t):
        return self.net(t)

In [40]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

In [41]:
class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, time_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, t_emb):
        h = self.block(x, t_emb)
        return self.pool(h), h

In [42]:
class Up(nn.Module):
    def __init__(self, in_ch, out_ch, skip_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 4, stride=2, padding=1)
        self.block = ResBlock(out_ch + skip_ch, out_ch, time_dim)

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.block(x, t_emb)


In [43]:
class SimpleUNet(nn.Module):
    def __init__(self, in_ch=1, base_ch=BASE_CH, time_dim=256):
        super().__init__()
        self.time_mlp = TimeMLP(time_dim, time_dim)
        self.in_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.down1 = Down(base_ch, base_ch, time_dim)
        self.down2 = Down(base_ch, base_ch * 2, time_dim)
        self.down3 = Down(base_ch * 2, base_ch * 4, time_dim)

        self.bottleneck1 = ResBlock(base_ch * 4, base_ch * 4, time_dim)
        self.bottleneck2 = ResBlock(base_ch * 4, base_ch * 4, time_dim)

        self.up3 = Up(base_ch * 4, base_ch * 2, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 2, base_ch, base_ch * 2, time_dim)
        self.up1 = Up(base_ch, base_ch, base_ch, time_dim)

        self.out_norm = nn.GroupNorm(8, base_ch)
        self.out_conv = nn.Conv2d(base_ch, in_ch, 3, padding=1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h = self.in_conv(x)

        d1, skip1 = self.down1(h, t_emb)
        d2, skip2 = self.down2(d1, t_emb)
        d3, skip3 = self.down3(d2, t_emb)

        b = self.bottleneck1(d3, t_emb)
        b = self.bottleneck2(b, t_emb)

        u3 = self.up3(b, skip3, t_emb)
        u2 = self.up2(u3, skip2, t_emb)
        u1 = self.up1(u2, skip1, t_emb)

        return self.out_conv(F.silu(self.out_norm(u1)))

In [44]:
class DDPM:
    def __init__(self, timesteps=T, beta_start=1e-4, beta_end=0.02, device=DEVICE):
        self.T = timesteps
        self.device = device

        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)

        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)

        self.posterior_variance = (
            self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.sqrt_alphas_cumprod[t][:, None, None, None]
        sqrt_1m_ac = self.sqrt_one_minus_alphas_cumprod[t][:, None, None, None]
        return sqrt_ac * x0 + sqrt_1m_ac * noise, noise

    @torch.no_grad()
    def p_sample(self, model, x_t, t_index):
        batch_size = x_t.shape[0]
        t = torch.full((batch_size,), t_index, device=self.device, dtype=torch.long)

        predicted_noise = model(x_t, t)

        beta_t = self.betas[t_index]
        alpha_t = self.alphas[t_index]
        sqrt_1m_ac_t = self.sqrt_one_minus_alphas_cumprod[t_index]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x_t - (beta_t / sqrt_1m_ac_t) * predicted_noise
        )

        if t_index == 0:
            return mean
        else:
            sigma_t = torch.sqrt(self.posterior_variance[t_index])
            z = torch.randn_like(x_t)
            return mean + sigma_t * z

    @torch.no_grad()
    def sample(self, model, shape, snapshot_steps=None, device=DEVICE):
        """
        Runs the full reverse diffusion process starting from pure Gaussian
        noise. If snapshot_steps is given, returns a dict {t: images} with
        the intermediate x_t at those timesteps (in addition to the final
        image), for visualization.
        """
        model.eval()
        x_t = torch.randn(shape, device=device)
        snapshots = {}
        snapshot_steps = set(snapshot_steps or [])

        if self.T in snapshot_steps:
            snapshots[self.T] = x_t.clone()

        for t_index in reversed(range(self.T)):
            x_t = self.p_sample(model, x_t, t_index)
            if t_index in snapshot_steps:
                snapshots[t_index] = x_t.clone()

        return x_t, snapshots

In [45]:
def train(model, ddpm, dataloader, epochs=EPOCHS, lr=LR, device=DEVICE):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []

    model.train()
    for epoch in range(epochs):
        epoch_loss, n_batches = 0.0, 0
        for images, _ in dataloader:
            images = images.to(device)
            batch_size = images.size(0)

            # b. randomly sample a diffusion timestep per image in the batch
            t = torch.randint(0, ddpm.T, (batch_size,), device=device).long()

            # forward-noise the images to x_t
            x_t, noise = ddpm.q_sample(images, t)

            # b. predict the noise with the U-Net
            predicted_noise = model(x_t, t)

            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        print(f"Epoch [{epoch + 1}/{epochs}]  MSE Loss: {avg_loss:.4f}")

    return losses

In [46]:
def plot_loss(losses, out_path=os.path.join(OUT_DIR, "ddpm_training_loss.png")):
    plt.figure(figsize=(7, 5))
    plt.plot(range(1, len(losses) + 1), losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("DDPM Training Loss vs. Epoch")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    print(f"Saved loss curve to {out_path}")

In [47]:
def to_display(img_tensor):
    img = img_tensor.detach().cpu().clamp(-1, 1)
    img = (img + 1) / 2
    return img.squeeze(0).numpy()

In [48]:
def visualize_denoising_steps(model, ddpm, n_rows=4, out_path=os.path.join(OUT_DIR, "reverse_diffusion_steps.png")):
    final_images, snapshots = ddpm.sample(
        model, shape=(n_rows, 1, IMG_SIZE, IMG_SIZE), snapshot_steps=SNAPSHOT_STEPS
    )

    fig, axes = plt.subplots(n_rows, len(SNAPSHOT_STEPS), figsize=(2.2 * len(SNAPSHOT_STEPS), 2.2 * n_rows))
    for row in range(n_rows):
        for col, step in enumerate(SNAPSHOT_STEPS):
            img = snapshots[step][row]
            ax = axes[row, col] if n_rows > 1 else axes[col]
            ax.imshow(to_display(img), cmap="gray")
            ax.axis("off")
            if row == 0:
                ax.set_title(f"t={step}")

    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    print(f"Saved reverse-diffusion step visualization to {out_path}")
    return final_images

In [49]:
def compare_with_real(generated_images, real_dataset, n_samples=4,
                       out_path=os.path.join(OUT_DIR, "generated_vs_real.png")):
    real_images = torch.stack([real_dataset[i][0] for i in range(n_samples)])

    fig, axes = plt.subplots(2, n_samples, figsize=(2.2 * n_samples, 4.4))
    for i in range(n_samples):
        axes[0, i].imshow(to_display(generated_images[i]), cmap="gray")
        axes[0, i].set_title("Generated")
        axes[0, i].axis("off")

        axes[1, i].imshow(to_display(real_images[i]), cmap="gray")
        axes[1, i].set_title("Real")
        axes[1, i].axis("off")

    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    print(f"Saved generated-vs-real comparison to {out_path}")
    return real_images

In [50]:
def evaluate_psnr_ssim(generated_images, real_images):
    n = min(len(generated_images), len(real_images))
    psnr_scores, ssim_scores = [], []

    for i in range(n):
        gen = to_display(generated_images[i])
        real = to_display(real_images[i])

        psnr_val = psnr_metric(real, gen, data_range=1.0)
        ssim_val = ssim_metric(real, gen, data_range=1.0)

        psnr_scores.append(psnr_val)
        ssim_scores.append(ssim_val)

    avg_psnr = float(np.mean(psnr_scores))
    avg_ssim = float(np.mean(ssim_scores))

    print(f"Average PSNR: {avg_psnr:.3f} dB")
    print(f"Average SSIM: {avg_ssim:.3f}")
    return avg_psnr, avg_ssim

In [51]:
dataloader, dataset = get_dataloader()

104385it [00:00, 954212.82it/s]


RuntimeError: File not found or corrupted.

In [ ]:
model = SimpleUNet(in_ch=1).to(DEVICE)
ddpm = DDPM(timesteps=T)

In [ ]:
losses = train(model, ddpm, dataloader)
plot_loss(losses)

In [ ]:
generated_images = visualize_denoising_steps(model, ddpm, n_rows=4)

In [ ]:
real_images = compare_with_real(generated_images, dataset, n_samples=4)

In [ ]:
evaluate_psnr_ssim(generated_images, real_images)

In [ ]:
torch.save(model.state_dict(), os.path.join(OUT_DIR, "ddpm_unet_emnist.pt"))